# Reproduce Public Metric Analysis

This notebook reproduces the core public analysis using de-texted metrics: entropy weighting, scoring comparison, fixed-effect style modelling, and clustering.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = "https://github.com/WindAlan-sw/luxury-brand-customer-engagement.git"
REPO_DIR = Path("luxury-brand-customer-engagement")

# In Colab, clone the repository if the public data folder is not already present.
if not Path("data/public_metrics").exists():
    try:
        if not REPO_DIR.exists():
            subprocess.run(["git", "clone", REPO_URL], check=True)
        os.chdir(REPO_DIR)
    except Exception as e:
        print("Could not clone repository. If the repo is private, upload the repository ZIP or make the repo public before using Colab.")
        print(e)

print("Working directory:", Path.cwd())
print("Public metrics folder exists:", Path("data/public_metrics").exists())


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.cluster.hierarchy import linkage, dendrogram
import statsmodels.formula.api as smf

DATA = Path("data/public_metrics")
post = pd.read_csv(DATA/"01_brand_post_metrics_detexted.csv")
entropy_input = pd.read_csv(DATA/"04_entropy_weight_input.csv")
score_methods = pd.read_csv(DATA/"06_scoring_method_post_results.csv")
fe = pd.read_csv(DATA/"11_fixed_effects_model_panel.csv")
cluster = pd.read_csv(DATA/"12_clustering_feature_matrix.csv")

## Entropy-weight input and scores

In [ ]:
metric_cols = ["retweet_count", "reply_count", "like_count", "quote_count"]
X = entropy_input[metric_cols].astype(float).clip(lower=0) + 1e-12
P = X / X.sum(axis=0)
k = 1 / np.log(len(X))
entropy = -k * (P * np.log(P)).sum(axis=0)
d = 1 - entropy
weights = d / d.sum()
weights = weights.rename("entropy_weight").reset_index().rename(columns={"index":"metric"})
display(weights)

## Scoring method comparison

In [ ]:
display(score_methods.describe().T)
method_means = score_methods.groupby("brand").mean(numeric_only=True).reset_index()
display(method_means)

## Fixed-effect style model on brand-month panel

In [ ]:
# This public model follows the notebook logic: brand-month sums of score and EITC variables.
model = smf.ols('score ~ Entertainment + Trendiness + Interaction + Customization + C(brand)', data=fe).fit()
print(model.summary())

## Hierarchical clustering

In [ ]:
features = cluster[["mean_CE_score", "CE_level", "mean_sentiment_score"]].to_numpy()
linked = linkage(features, method="ward")
plt.figure(figsize=(8,5))
dendrogram(linked, labels=cluster["brand"].tolist())
plt.ylabel("Euclidean distance")
plt.title("Hierarchical clustering from public feature matrix")
plt.tight_layout()
plt.show()